# TKCE — Colab Pro+ run (resumable)

**Before you start:**
1. `Runtime → Change runtime type →` **GPU (T4 or L4)** + **High-RAM**.
2. Turn on **background execution** (Pro+) so it keeps running if you close the tab.
3. Edit `REPO_URL` in the CONFIG cell to your GitHub repo.

**Everything is saved to Google Drive as it runs.** If the session dies, just
re-run **Clone+Install → GPU check → Full run**; the suite skips whatever already
finished and continues. Nothing is lost.

### Closing the browser / running while you're away
On **Pro+** the run keeps going after you **close the tab or shut your laptop** —
it runs in the cloud (background execution, up to **24 h** per session). Just make
sure the **Full run cell is already executing** before you close it, then reopen
the notebook later to reconnect and watch progress. You do **not** need to keep
your computer on.

**Note:** the full run (~35–40 h) is longer than the 24 h session cap, so it will
pause around the 24 h mark. That is expected — reopen, re-run
**Clone+Install → GPU → Full run**, and it resumes. Because results are written to
Drive after every model, you never lose more than the single model in progress.

## 1. Config — edit these two lines

In [1]:
REPO_URL   = "https://github.com/sushanedulloo/TKCE.git"   # this project's public repo
DRIVE_BASE = "/content/drive/MyDrive/tkce_results"         # everything is saved here

## 2. Mount Google Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs(DRIVE_BASE, exist_ok=True)
print('Saving everything to:', DRIVE_BASE)

Mounted at /content/drive
Saving everything to: /content/drive/MyDrive/tkce_results


## 3. Clone the code + install (re-run this after any disconnect)

In [3]:
%cd /content
!rm -rf tkce_repo
!git clone $REPO_URL tkce_repo
%cd tkce_repo
!pip install -q -r requirements-tkce.txt
print('Repo + dependencies ready')

/content
Cloning into 'tkce_repo'...
remote: Enumerating objects: 39, done.
remote: Counting objects: 100% (39/39), done.
remote: Compressing objects: 100% (34/34), done.
remote: Total 39 (delta 3), reused 39 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (39/39), 75.54 KiB | 25.18 MiB/s, done.
Resolving deltas: 100% (3/3), done.
/content/tkce_repo
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 28.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.4/160.4 kB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 40.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 30.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.8/93.8 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 106.7 MB/s eta 0:00:00
Repo + dependencies ready


## 4. GPU check

In [4]:
import torch
print('CUDA available:', torch.cuda.is_available())
!nvidia-smi -L

CUDA available: True
GPU 0: NVIDIA L4 (UUID: GPU-24b201ac-ee53-4a4b-3093-3b5e334b6f17)


## 5. (Optional but recommended) Time ONE dataset first
Run this once to get a real per-dataset time, then multiply by 15. It also counts
as progress (same output folder style), so nothing is wasted.

In [ ]:
!time python run_suite.py --tasks 361070 --seeds 0,1,2 --trials 100 --max-rows 16000 --device auto --out "$DRIVE_BASE/probe" 2>&1 | tee "$DRIVE_BASE/probe_run.log"

## 6. Full run — 15 datasets, resumable, saves to Drive
Re-run this cell to resume after a disconnect: it reads the results file on Drive
and skips every dataset×model cell that already finished. The `tee` command also
appends a live log to Drive so you can see progress at any time.

In [ ]:
TASKS = "361070,361278,361062,361066,361280,361076,361072,361279,361286,361283,361110,361093,361094,361098,361287"
SUITE_OUT = DRIVE_BASE + "/suite_v1"
LOG = DRIVE_BASE + "/suite_v1_run.log"
!python run_suite.py --tasks $TASKS --seeds 0,1,2 --trials 100 --max-rows 16000 --device auto --reference catboost --out "$SUITE_OUT" 2>&1 | tee -a "$LOG"

## 7. Check progress any time

In [ ]:
# Target rows = 16 models x 3 seeds x 15 datasets = 720 (approx; some cells may be skipped/failed)
!wc -l "$SUITE_OUT/results_long.csv" 2>/dev/null || echo 'no results yet'
!echo '--- last log lines ---'
!tail -n 10 "$LOG" 2>/dev/null

### If the session disconnects
Re-run **cell 3 (clone+install)**, **cell 4 (GPU)**, then **cell 6 (full run)**.
It continues from where it stopped. You can re-run cell 6 as many times as needed.

## 8. Build the paper report (run any time, even on partial results)
Produces the main table, gap analysis, two-stage-vs-joint, kernel ablation,
average-rank table, the critical-difference figure, and Wilcoxon tests.

In [ ]:
!python make_report.py --results "$SUITE_OUT/results_long.csv" --out "$SUITE_OUT/report"
!echo '--- average rank (lower = better) ---'
!cat "$SUITE_OUT/report/avg_rank.csv" 2>/dev/null

## 9. Analysis experiments (figures for the paper)
These are shorter than the main suite. Figures + CSVs save to Drive.

In [ ]:
FIG = DRIVE_BASE + "/figures"
AN  = DRIVE_BASE + "/analysis"
!python run_loss_ablation.py   --seeds 0,1,2 --device auto --out "$FIG" --csv "$AN"
!python run_mechanism.py       --seeds 0,1,2 --device auto --out "$FIG" --csv "$AN"
!python run_data_efficiency.py --seeds 0,1,2 --device auto --out "$FIG" --csv "$AN"
!python run_lambda_sweep.py    --seeds 0,1,2 --device auto --out "$FIG" --csv "$AN"

## 10. See everything that was saved

In [ ]:
!echo 'Files in your Drive results folder:'
!ls -R "$DRIVE_BASE" | head -80